# SUMMA on eWaterCycle

This notebook demonstrates the [ewatercycle-summa](https://github.com/DarriEy/ewatercycle-summa) plugin, which brings [SUMMA](https://github.com/CH-Earth/summa) (Structure for Unifying Multiple Modeling Alternatives) into the [eWaterCycle](https://ewatercycle.readthedocs.io/) platform.

## What this plugin provides

- **SUMMA as a first-class eWaterCycle model** — discoverable via `ewatercycle.models.sources["SUMMA"]`
- **True time-stepping** — restart-based BMI, each `update()` advances one forcing timestep (~0.5s/step)
- **Containerized execution** — SUMMA runs inside Docker via [grpc4bmi](https://github.com/eWaterCycle/grpc4bmi), no local Fortran compilation needed
- **Multi-arch container** — `ghcr.io/darriey/summa-grpc4bmi:v0.1.0` (linux/amd64 + linux/arm64)
- **Parameter set generation** — create a complete SUMMA setup from a shapefile + DEM + rasters (requires [SYMFLUENCE](https://github.com/DarriEy/SYMFLUENCE))

## Requirements

```bash
pip install ewatercycle-summa
# Docker must be running — the container is pulled automatically on first use
```

## 1. Plugin Discovery

Once installed, SUMMA appears automatically in eWaterCycle's model registry.

In [ ]:
from ewatercycle.models import sources

print("Available models:", list(sources.keys()))
assert "SUMMA" in sources, "ewatercycle-summa not installed!"

SUMMA = sources["SUMMA"].load()
print(f"SUMMA class: {SUMMA}")
print(f"Container image: {SUMMA.model_fields['bmi_image'].default}")

## 2. Create a Parameter Set

A SUMMA parameter set contains 12+ configuration files. You can either:

- **Use an existing domain** from [SYMFLUENCE](https://github.com/DarriEy/SYMFLUENCE) (recommended)
- **Generate from scratch** using `create_parameter_set()` (requires `symfluence`)
- **Point to any standard SUMMA setup** directory

Here we use the Bow River at Banff domain — a lumped catchment in the Canadian Rockies with RDRS forcing (2002-2009, hourly).

> **Note:** Update the `DOMAIN_DIR` path below to point to your SUMMA domain.

In [ ]:
from pathlib import Path
from ewatercycle.base.parameter_set import ParameterSet

# Point to an existing SUMMA domain
DOMAIN_DIR = Path.home() / "compHydro/SYMFLUENCE_data/domain_Bow_at_Banff_lumped"

parameter_set = ParameterSet(
    name="bow_at_banff",
    directory=DOMAIN_DIR,
    config="settings/SUMMA/fileManager.txt",
    target_model="SUMMA",
)

print(f"Parameter set: {parameter_set.name}")
print(f"Directory: {parameter_set.directory}")
print(f"Config: {parameter_set.config}")

## 3. Create and Configure the Model

The SUMMA model parses `fileManager.txt` on creation and exposes simulation times.

In [ ]:
model = SUMMA(parameter_set=parameter_set)

print(f"Simulation period: {model.start_time_as_datetime} to {model.end_time_as_datetime}")
print(f"Configuration keys: {list(dict(model.parameters).keys())}")

## 4. Setup and Initialize

`setup()` generates the runtime configuration, copies settings, and starts the Docker container with grpc4bmi.

You can override the simulation window with `start_time` and `end_time`.

In [ ]:
import tempfile

# Use a temporary directory for this run
work_dir = tempfile.mkdtemp(prefix="summa_ewc_")

# Run a 1-week window (override the full 8-year period)
cfg_file, cfg_dir = model.setup(
    cfg_dir=work_dir,
    start_time="2002-06-01 01:00",
    end_time="2002-06-08 01:00",
)

model.initialize(cfg_file)

print(f"Current model time: {model.time_as_datetime}")
print(f"Working directory: {cfg_dir}")

## 5. Run the Model (Time-Stepping)

Each `update()` call runs SUMMA for one forcing timestep (typically 1 hour) using restart file chaining. This gives true BMI time-stepping — you can read output after each step.

In [ ]:
import time
import numpy as np

n_steps = 48  # 2 days of hourly steps
times = []
swe = []
canopy_water = []

print(f"Running {n_steps} hourly timesteps...")
t0 = time.time()

for i in range(n_steps):
    model.update()
    times.append(model.time_as_datetime)
    
    # Read output variables
    val = np.zeros(1)
    model._bmi.get_value("scalarSWE", val)
    swe.append(val[0])
    
    model._bmi.get_value("scalarCanopyWat", val)
    canopy_water.append(val[0])

elapsed = time.time() - t0
print(f"Done: {elapsed:.1f}s total, {elapsed/n_steps:.2f}s/step")
print(f"Time range: {times[0]} to {times[-1]}")
print(f"\nAvailable output variables:")
print(model._bmi.get_output_var_names())

## 6. Visualize Results

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

ax1.plot(times, swe, "b-", linewidth=1.5)
ax1.set_ylabel("Snow Water Equivalent (kg/m²)")
ax1.set_title("SUMMA via eWaterCycle — Bow River at Banff")
ax1.grid(True, alpha=0.3)

ax2.plot(times, canopy_water, "g-", linewidth=1.5)
ax2.set_ylabel("Canopy Water (kg/m²)")
ax2.set_xlabel("Time")
ax2.grid(True, alpha=0.3)

ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b %d\n%H:%M"))
fig.tight_layout()
plt.show()

## 7. Clean Up

In [ ]:
model.finalize()
print("Model finalized, container stopped.")

---

## Appendix: Generating a Parameter Set from Scratch

If you have raw geospatial data (catchment shapefile, DEM, soil/land cover rasters) and SUMMA-format forcing, you can generate a complete parameter set using SYMFLUENCE:

```bash
pip install ewatercycle-summa[generate]
```

```python
from ewatercycle_summa.parameter_set import create_parameter_set

create_parameter_set(
    domain_name="my_basin",
    shapefile="/path/to/catchment.shp",
    dem="/path/to/dem.tif",
    soil_raster="/path/to/soil_classes.tif",
    landclass_raster="/path/to/land_cover.tif",
    forcing_dir="/path/to/forcing/SUMMA_input/",
    start_time="2000-01-01 00:00",
    end_time="2005-01-01 00:00",
    output_dir="/path/to/output_parameter_set/",
)
```

This creates all 12 required SUMMA files (attributes.nc, coldState.nc, trialParams.nc, fileManager.txt, model decisions, lookup tables, etc.) with real soil and land cover classes extracted from the rasters.

## Architecture

```
ewatercycle-summa plugin
├── model.py          eWaterCycle model class (ContainerizedModel)
├── forcing/           ESMValTool forcing generation + diagnostic script
├── parameter_set.py   Parameter set generation (wraps SYMFLUENCE)
└── container/         Docker: standard SUMMA exe + Python BMI wrapper + grpc4bmi

Execution flow:
  setup() → writes fileManager.txt, starts Docker container
  initialize() → BMI wrapper parses config inside container
  update() → runs SUMMA for 1 timestep, writes restart, reads output
  get_value() → returns output from current timestep
  finalize() → stops container
```

## Links

- **Plugin**: [github.com/DarriEy/ewatercycle-summa](https://github.com/DarriEy/ewatercycle-summa)
- **PyPI**: [pypi.org/project/ewatercycle-summa](https://pypi.org/project/ewatercycle-summa/)
- **Container**: `ghcr.io/darriey/summa-grpc4bmi:v0.1.0`
- **SUMMA**: [github.com/CH-Earth/summa](https://github.com/CH-Earth/summa)
- **SYMFLUENCE**: [github.com/DarriEy/SYMFLUENCE](https://github.com/DarriEy/SYMFLUENCE)
- **eWaterCycle**: [ewatercycle.readthedocs.io](https://ewatercycle.readthedocs.io/)